This notebook demonstrates how to use the clemcore Pettingzoo integration to run a
multi-player game with the game **"Taboo"**, and how to plug in a self-hosted
OpenAI-compatible model server as an agent backend.

You will learn how to:

1. Set up the environment and install dependencies.
2. Register and configure a model as an agent backend.
3. Implement a simple custom describer agent.
4. Run an automated gameplay session and inspect the interactions.

The main difference to the Gymnasium-style API is that both players' turns in the loop are under your control.
This is also necessary for self-play.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/phisad/playpen/blob/main/examples/pettingzoo/taboo.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"taboo"`, but this pattern works for any 2‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [1]:
import os

# Specify the game name here (this code can be adapted to any 2-player game)
GAME_NAME = "taboo"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the `taboo` game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `taboo` listed by `clem list games -s taboo`.

If you do **not** see `taboo`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [ ]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

In [15]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

clem 3.3.5
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'taboo'}'
taboo:
 	Taboo game between two agents where one has to describe a word for
	the other to guess.


# 2. Register and configure a model backend

We will use a self-hosted OpenAI-compatible model server and register the model
under the name `"clp-chat"`.

This name (`"clp-chat"`) will later be used when we assign the model to a player
in the PettingZoo-style environment.

In [2]:
# Register the remote model under the name "clp-chat"
!clem register model -n "clp-chat" -v "backend=openai_compatible,model_id=Qwen/Qwen3-VL-30B-A3B-Thinking-FP8" --cwd

Updated model registry at /Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json successfully: {"model_name":"clp-chat","backend":"openai_compatible","lookup_source":"/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json","model_id":"Qwen/Qwen3-VL-30B-A3B-Thinking-FP8"}


You can achieve the same registration programmatically if you prefer not to use
the CLI:

In [3]:
from clemcore.backends import ModelRegistry

registry = ModelRegistry.register("clp-chat", backend="openai_compatible", model_id="Qwen/Qwen3-VL-30B-A3B-Thinking-FP8")
registry.get_first_model_spec_that_unify_with("clp-chat")

ModelSpec({'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/pettingzoo/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Thinking-FP8'})

## 2.2. Register API key and base URL

To connect to your OpenAI-compatible model server, we also register:

- An API key
- An organisation (if applicable)
- A base URL for the endpoint

Replace the placeholders with your actual values.

In [6]:
API_KEY = "<insert-api-key-here>"
ORGANIZATION = "<insert-organisation-here>"
BASE_URL = "<insert-base-url-here>"

In [ ]:
# Register the credentials and base URL for the "openai_compatible" backend
!clem register key -n "openai_compatible" -v "api_key=$API_KEY,organisation=$ORGANIZATION,base_url=$BASE_URL" --cwd

Programmatic equivalent:

In [ ]:
from clemcore.backends import KeyRegistry

KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

# 3. Implement a custom describer agent

Next, we implement a simple custom describer agent by subclassing `ClemAgent`.

In this example:

- The **describer** (`MyAgenticDescriber`) calls the registered `"clp-chat"` model
  on the full interaction history and uses the model output as its next action.
- The **guesser** (`my_guesser_agent`) is a trivial baseline that randomly
  guesses between `"apples"` and `"bananas"` to illustrate how non-LLM agents
  can also be plugged in.

This is just a toy setup to show how you can integrate both LLM-backed agents
and hand-written Python policies into the same PettingZoo-style environment.

In [4]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model


def my_guesser_agent(_):
    import random
    random_word = random.choice(["apples", "bananas"])
    response = f"GUESS: {random_word}"
    return response


class MyAgenticDescriber(ClemAgent):
    """
    Simple example agent for Taboo.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.7, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe our own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text


describer = MyAgenticDescriber()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'

2026-01-09 12:59:27,263 - clemcore.backends - INFO - Found registered model spec that unifies 

# 4. Run an automated gameplay session

We now:

1. Create a PettingZoo-style environment for the game.
2. Use our custom describer as `player_0` and a dummy guesser as `player_1`.
3. Run through a single episode step-by-step using the PettingZoo API.
4. Automatically run through the whole episode and log the interaction at each step.

The environment follows the standard PettingZoo interaction pattern:

- `env.agent_iter()` yields one `agent_id` at a time.
- `env.last()` returns the `(observation, reward, termination, truncation, info)`
  for the **current** agent.
- `env.step(action)` submits an action for the current agent and advances the game.

`termination` becomes `True` when the underlying game episode is logically over.
`truncation` becomes `True` if the episode ends early due to a time limit or
other external cutoff.

Setting `single_pass=False` (the default) allows iterating through all available
game instances multiple times. You can restrict which instances are used via the
`game_instance_filter` argument of the `env()` function.

In [5]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clp-chat", result_dir_path="playpen-records", player_model_infos="MyAgenticDescriber")

game_env = env(
    "taboo",
    game_instance_filter=lambda game_name, experiment_name: [1, 5, 10],
    single_pass=False,
    callbacks=callbacks
)
game_env.reset()

2026-01-09 13:03:10,508 - clemcore.cli - INFO - Found '1' game matching the game_selector="taboo"
2026-01-09 13:03:10,510 - clemcore.cli - INFO - {
  "game_name": "taboo",
  "description": "Taboo game between two agents where one has to describe a word for the other to guess.",
  "main_game": "taboo",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0"
  ],
  "regression": "small",
  "roles": [
    "Describer",
    "Guesser"
  ],
  "game_path": "/Users/philippsadler/git/clembench/taboo"
}
2026-01-09 13:03:10,511 - clemcore.run - INFO - Loading game benchmark for taboo
2026-01-09 13:03:10,798 - clemcore.run - INFO - Loading game benchmark for taboo took: 0:00:00.286581
2026-01-09 13:03:10,802 - clemcore.run - INFO - Sub-select for taboo experiment high_en instances with game_ids: [1, 5, 10]
2026-01-09 13:03:10,803 - clemcore.run - INFO - Sub-select for taboo experiment medium_en instances with game_ids: [1, 5, 10]

In [6]:
# Let's peek at the possible player ids (only available after reset, because reset() initiates the GameMaster)
print("possible agents:", game_env.possible_agents)
# In most cases, roles will be the content description of what player_0 and player_1 are
print("likely mapping:", game_env.unwrapped.game_master.game_spec["roles"])
agent_mapping = {"player_0": describer, "player_1": my_guesser_agent}

possible agents: ['player_0', 'player_1']
likely mapping: ['Describer', 'Guesser']


In [9]:
# Now we can iterate through the agent turns, one step at a time (calling this cell multiple times)

try:
    agent_id = next(iter(game_env.agent_iter()))
    print("Agent_id:", agent_id)
    context, reward, termination, truncation, info = game_env.last()
    print("Reward:", reward, "Termination:", termination, "Truncation:", truncation, "Info:", info)
    if termination or truncation:
        response = None # we step one more time to remove the agent from the env (final reward was observed in last)
    else:
        response = agent_mapping[agent_id](context)
    print("Context:", context)
    print("Response:", response)
    game_env.step(response)
except StopIteration:
    print("Episode finished.")
    describer.reset()

Agent_id: player_0
Reward: 0 Termination: False Truncation: False Info: {'response_score': 0, 'response_feedback': None, 'episode_score': 0}
Context: {'role': 'user', 'content': 'GUESS: apples'}
Response: CLUE: self-sufficient


In [10]:
# Now we can do everything automated as well
game_env.reset()
context_response_pairs = []
for agent_id in game_env.agent_iter():
    context, reward, termination, truncation, info = game_env.last()
    if termination or truncation:
        response = None # we step one more time to remove the agent from the env (final reward was observed in last)
    else:
        response = agent_mapping[agent_id](context)
    context_response_pairs.append((agent_id, context, response, reward))
    game_env.step(response)

describer.reset()

print(f"Episode took these {len(context_response_pairs)} steps:")
print("-" * 20)
for idx, (agent_id, context, response, reward) in enumerate(context_response_pairs):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"Agent({agent_id}) <- Context:", context)
    print(f"Agent({agent_id}) -> Response:", response)
    print("-" * 20)

# Note that there is a last None-action step to collect any rewards that happened between the player's turns.
# For example, when the guesser guesses the word on its final turn, then also the describer should receive the positive game reward; if both cannot find the word until the last turn, then also the describer receives a negative game reward, although the guesser makes the last turn.
# You have to decide whether
# - to merge them with the second last observation
# - to ignore them (because the other player caused to abort the game, or was incapable of guessing an obvious word)
# - to keep them as a separate observation with a None action

Episode took these 8 steps:
--------------------
Step 0 / Reward 0.00:
Agent(player_0) <- Context: {'role': 'user', 'content': 'You are playing a collaborative word guessing game in which you have to describe a target word for another player to guess.\n\nRules:\n(a) You have to reply in the form: CLUE: <some text>. Guesses from the other player will start with GUESS.\n(b) You cannot use the target word itself, parts or morphological variants of it in your description.\n(c) In addition, the same rules apply for related words which are provided below.\n\nEnd conditions:\n(i) If you use the target word or a related word in your description, then you lose.\n(ii) If the other player can guess the target word in 3 tries, you both win.\n\nLet us start.\n\nThis is the target word that you need to describe and that the other player needs to guess:\n\nenergy\n\nRelated words are:\n\n- solar\n- power\n- electricity\n\nImportant: You are under time pressure, give short descriptions that are to the